# 02. Data Analysis & Visualization
descrizione

### Indice Capitoli
- **Prefazione: Data Loading** Caricamento dei 13 dataset e verifica dtypes.
- **Capitolo 1: Analisi Temporale**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
%load_ext autoreload
%autoreload 2
from functions.utils import *

##  Prefazione: Data Loading

Iniziamo caricando i dataset con cui lavoreremo, Pandas non legge i tipi converiti precedentemente quindi li convertiremo di nuovo per una corretta analisi

In [ ]:
characters_df = pd.read_csv("../data/processed/Characters_cleaned.csv")
character_anime_works_df = pd.read_csv("../data/processed/Character_anime_works_cleaned.csv")
character_nicknames_df = pd.read_csv("../data/processed/Character_nicknames_cleaned.csv")
details_df = pd.read_csv("../data/processed/Details_cleaned.csv")
favs_df = pd.read_csv("../data/processed/Favs_cleaned.csv")
person_alternate_names_df = pd.read_csv("../data/processed/Person_alternate_names_cleaned.csv")
person_anime_works_df = pd.read_csv("../data/processed/Person_anime_works_cleaned.csv")
person_voice_works_df = pd.read_csv("../data/processed/Person_voice_works_cleaned.csv")
person_details_df = pd.read_csv("../data/processed/Person_details_cleaned.csv")
profiles_df = pd.read_csv("../data/processed/Profiles_cleaned.csv")
ratings_df = pd.read_csv("../data/processed/Ratings_cleaned.csv")
recommendations_df = pd.read_csv("../data/processed/Recommendations_cleaned.csv")
stats_df = pd.read_csv("../data/processed/Stats_cleaned.csv")

In [ ]:
datasets = {
    "Characters": characters_df,
    "Character_anime_works": character_anime_works_df,
    "Character_nicknames": character_nicknames_df,
    "Details": details_df,
    "Favs": favs_df,
    "Person_alternate_names": person_alternate_names_df,
    "Person_anime_works": person_anime_works_df,
    "Person_voice_works": person_voice_works_df,
    "Person_details": person_details_df,
    "Profiles": profiles_df,
    "Ratings": ratings_df,
    "Recommendations": recommendations_df,
    "Stats": stats_df
}

for name, df in datasets.items():
     show_shape(df, name)

In [ ]:
for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    print(df.dtypes)

In [ ]:
datasets["Person_details"] = convert_to_datetime(datasets["Person_details"], "birthday")

datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "joined")
datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "birthday")

datasets["Details"] = convert_to_datetime(datasets["Details"], "start_date")
datasets["Details"] = convert_to_datetime(datasets["Details"], "end_date")

In [ ]:
datasets["Details"]["year"] = datasets["Details"]["year"].astype("Int64")
datasets["Details"]["scored_by"] = datasets["Details"]["scored_by"].astype("Int64")
datasets["Details"]["rank"] = datasets["Details"]["rank"].astype("Int64")
datasets["Details"]["episodes"] = datasets["Details"]["episodes"].astype("Int64")
datasets["Characters"]["favorites"] = datasets["Characters"]["favorites"].astype("Int64")
cols_to_fix = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']

for col in cols_to_fix:
    if col in datasets["Profiles"].columns:
        datasets["Profiles"][col] = datasets["Profiles"][col].astype("Int64")

## Capitolo 1: Analisi Temporale

### Analisi della produzione annuale

Dando un’occhiata alla produzione annuale, si capisce subito come l’animazione giapponese sia cresciuta nel tempo.
`Nota`: ho escluso il 2026 perché è l’anno in corso: i dati sono ancora parziali e avrebbero mostrato un calo che, in realtà, non è significativo.

- **Dal 1961 al 1997** la crescita è costante ma lenta. Produrre anime era un lavoro complesso, poiché non esistevano i software moderni e i costi di produzione erano molto elevati.
- **Dopo il 1998** arriva il digitale e la produzione aumenta sensibilmente. Escono titoli come Cowboy Bebop e One Piece, mentre l’influenza di anime iconici come Sailor Moon, Dragon Ball ed Evangelion, prodotti nel decennio precedente, hanno contribuito a rendere l’industria più attrattiva e sostenibile.
- **Tra il 2007 e il 2010** si osserva un lieve calo, probabilmente dovuto sia al declino del mercato fisico dei DVD, da cui l’industria degli anime dipendeva fortemente, sia alla crisi economica del 2008, che ha portato a una riduzione dei budget.
- **Dopo il 2012** si registra una forte crescita, favorita dall’espansione dello streaming globale grazie a piattaforme come Netflix e Crunchyroll.
- **Tra il 2019 e il 2022** è presente un nuovo calo: la pandemia di COVID-19 ha causato cancellazioni e ritardi produttivi, mentre la sovrapproduzione degli anni precedenti ha spinto l’industria a ridurre il numero di titoli.
- **Negli ultimi tre anni**, infine, la produzione si è mantenuta relativamente stabile.

In [ ]:
plot_production_volume(details_df)

### Analisi della produzione annuale per Source 

Il grafico riportato sotto rappresenta la produzione di anime suddivisa per **Source**, ovvero il materiale di partenza degli adattamenti anime.  
La categoria "Unknown" è stata esclusa, in quanto non rilevante ai fini dell’analisi.

Si nota subito una competizione tra **Manga** e **Original**:  
- **Dagli anni '80 fino al 2010**, gli anime adattati da manga sono stati predominanti, pur mantenendo un distacco contenuto rispetto agli anime originali.  
- **Dal 2010 al 2022**, gli anime originali registrano un picco significativo, superando le altre fonti in termini di numero di produzioni.
- **Negli ultimi due anni** si osserva però un’inversione di tendenza:  
  - Diminuiscono gli anime originali, mentre aumentano in maniera esponenziale gli adattamenti da manga.  
  - Questo trend è dovuto principalmente alla maggiore sicurezza economica offerta dall’adattamento di opere già pubblicate, che possiedono una fanbase consolidata.
  
Un altro fenomeno rilevante riguarda le **light novel**, che compaiono più tardi rispetto ai videogiochi come fonte di adattamento.  
- A partire dal 2020, gli adattamenti da light novel hanno registrato una crescita notevole.  
- Come i manga, le light novel offrono un pubblico già consolidato, riducendo il rischio finanziario per le case di produzione.  
- Inoltre, il boom di generi virali, come l’**isekai**, contribuisce in modo significativo alla loro popolarità negli adattamenti anime.

In [ ]:
plot_production_evolution_by_source(details_df)

`Nota`: In questa analisi temporale riportata sopra sono stati esclusi i titoli il cui anno di rilascio risulta NaN. Per questo motivo, il grafico potrebbe dare l'impressione che gli adattamenti Manga siano la fonte più popolare negli anni. Tuttavia, adottando un'analisi volumetrica, emerge chiaramente che gli Original superano in realtà di più del doppio gli adattamenti Manga.

In [ ]:
plot_source_popularity(details_df)

### Analisi 1.3

In [ ]:
plot_genre_production_by_year(details_df)